# Bulk Join: All Shared League-Seasons

Phase 1 — Data Inspection, part 6. This notebook runs the approved src.join method over every season present in both sources and produces one player-season-price table — the handover's core deliverable (performance keys + market price, per player per season).

Scope rule (leagues): StatsBomb seasons starting 2012–2023 for La Liga, Premier League, Bundesliga, Serie A, Ligue 1, and Champions League — Transfermarkt covers all these spans. Cups and internationals (Euros, World Cups, Copa América) are listed explicitly with date ranges, because Transfermarkt's season label is not the event year there (proven in Cell 2). Skipped for zero overlap: Europa League (only 1988/89 locally), Copa del Rey (only 1970s–80s), MLS (years miss), pre-2006 World Cups.

Time note: around 3,000 lineup files plus per-season matching. Expect 30–45 minutes for Cells 4–8. Run Cells 1–3 first (fast), check the scope table, then run the rest and step away.

## Cell 1 — Setup, imports, and cup list

Matching logic comes only from src.join. League scope is computed by rule in Cell 3; the five cup seasons are listed here with verified date ranges (event dates, never Transfermarkt season labels).

In [1]:
import json, os, re, sys
import pandas as pd
import numpy as np

ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, ROOT)
from src.join import CLUB_MAP, NICKNAMES, normalize_name, resolve_sample

TM_DIR = os.path.join(ROOT, 'data', 'transfermarkt')
SB_DATA_DIR = os.path.join(ROOT, 'data', 'statsbomb', 'data')
OUT_DIR = os.path.join(ROOT, 'data', 'processed')

# (sb_comp, sb_season_id, tm_comp, window_start, window_end, label)
TOURNAMENTS = [
    (55, 43, 'EURO', '2021-06-01', '2021-07-31', 'Euro 2020'),
    (55, 282, 'EURO', '2024-06-01', '2024-07-31', 'Euro 2024'),
    (43, 3, 'FIWC', '2018-06-01', '2018-07-31', 'World Cup 2018'),
    (43, 106, 'FIWC', '2022-11-01', '2022-12-31', 'World Cup 2022'),
    (223, 282, 'COPA', '2024-06-01', '2024-07-31', 'Copa America 2024'),
]
LEAGUE_COMPS = {11, 2, 9, 12, 7, 16}
print('root:', ROOT)


root: c:\Users\User\Development\football-recruitment-intelligence


## Cell 2 — Proof that cups need date ranges

Transfermarkt files tournaments under the prior year (Euro 2020 under season 2020 with 2021 dates; World Cup 2018 under 2017). This cell prints the actual date ranges so the Cell 1 dates are evidenced, not trusted. If any range disagrees, fix the cup list before continuing.

In [2]:
games = pd.read_csv(os.path.join(TM_DIR, 'games.csv'), usecols=['competition_id', 'season', 'date'], parse_dates=['date'])
for cid in ['EURO', 'FIWC', 'COPA']:
    sub = games[games['competition_id'] == cid]
    print('##', cid)
    print(sub.groupby('season')['date'].agg(['min', 'max']).to_string())
    print()
sb_comp = pd.read_json(os.path.join(SB_DATA_DIR, 'competitions.json'))
mask = sb_comp.set_index(['competition_id', 'season_id']).index.isin(
    [(c, s) for c, s, _, _, _, _ in TOURNAMENTS])
print(sb_comp[mask][['competition_name', 'season_name']].to_string())


## EURO
              min        max
season                      
2007   2008-06-07 2008-06-29
2011   2012-06-08 2012-07-01
2015   2016-06-10 2016-07-10
2020   2021-06-11 2021-07-11
2023   2024-06-14 2024-07-14

## FIWC
              min        max
season                      
2005   2006-06-09 2006-07-09
2009   2010-06-11 2010-07-11
2013   2014-06-12 2014-07-13
2017   2018-06-14 2018-07-15
2021   2022-11-20 2022-12-18
2025   2026-06-11 2026-06-28

## COPA
              min        max
season                      
2025   2024-06-21 2024-07-15

   competition_name season_name
21     Copa America        2024
30   FIFA World Cup        2022
31   FIFA World Cup        2018
73        UEFA Euro        2024
74        UEFA Euro        2020


## Cell 3 — Full scope table (check before the big run)

League rows come from competitions.json by rule (start-year 2012–2023); cup rows come from Cell 1. Every row is one matching job. Check the printed table: one wrong row multiplies into a wrong season downstream.

In [3]:
sb_comp = pd.read_json(os.path.join(SB_DATA_DIR, 'competitions.json'))

def start_year(season_name):
    m = re.match(r'(\d{4})', str(season_name))
    return int(m.group(1)) if m else None

sb_comp['start_yr'] = sb_comp['season_name'].apply(start_year)
leagues = sb_comp[(sb_comp['competition_id'].isin(LEAGUE_COMPS))
                  & (sb_comp['start_yr'] >= 2012) & (sb_comp['start_yr'] <= 2023)]
TM_COMP_OF = {11: 'ES1', 2: 'GB1', 9: 'L1', 12: 'IT1', 7: 'FR1', 16: 'CL'}
scope = []
for _, r in leagues.iterrows():
    y = r['start_yr']
    scope.append((r['competition_id'], r['season_id'], TM_COMP_OF[r['competition_id']],
                  f'{y}-07-01', f'{y + 1}-06-30',
                  f"{r['competition_name']} {r['season_name']}"))
for c, s, tm, a, b, label in TOURNAMENTS:
    scope.append((c, s, tm, a, b, label))
SCOPE = pd.DataFrame(scope, columns=['sb_comp', 'sb_season', 'tm_comp', 'start', 'end', 'label'])
print('scope rows:', len(SCOPE))
print(SCOPE[['label', 'tm_comp', 'start', 'end']].to_string())


scope rows: 28
                         label tm_comp       start         end
0      1. Bundesliga 2023/2024      L1  2023-07-01  2024-06-30
1      1. Bundesliga 2015/2016      L1  2015-07-01  2016-06-30
2   Champions League 2018/2019      CL  2018-07-01  2019-06-30
3   Champions League 2017/2018      CL  2017-07-01  2018-06-30
4   Champions League 2016/2017      CL  2016-07-01  2017-06-30
5   Champions League 2015/2016      CL  2015-07-01  2016-06-30
6   Champions League 2014/2015      CL  2014-07-01  2015-06-30
7   Champions League 2013/2014      CL  2013-07-01  2014-06-30
8   Champions League 2012/2013      CL  2012-07-01  2013-06-30
9            La Liga 2020/2021     ES1  2020-07-01  2021-06-30
10           La Liga 2019/2020     ES1  2019-07-01  2020-06-30
11           La Liga 2018/2019     ES1  2018-07-01  2019-06-30
12           La Liga 2017/2018     ES1  2017-07-01  2018-06-30
13           La Liga 2016/2017     ES1  2016-07-01  2017-06-30
14           La Liga 2015/2016     ES1  

## Cell 4 — Bulk StatsBomb side (slow: around 3,000 lineup files)

One row per player-team-season. Files contain whatever StatsBomb ships (older seasons are partial samples, not full 380-game seasons) — the loop processes what exists, and Cell 6's per-season counts show the denominators.

In [4]:
sb_rows = []
for _, r in SCOPE.iterrows():
    fp = os.path.join(SB_DATA_DIR, 'matches', str(r['sb_comp']), f"{r['sb_season']}.json")
    for m in json.load(open(fp, encoding='utf-8')):
        lfp = os.path.join(SB_DATA_DIR, 'lineups', f"{m['match_id']}.json")
        for t in json.load(open(lfp, encoding='utf-8')):
            for p in t['lineup']:
                sb_rows.append((r['label'], p['player_name'], t['team_name']))
SB_ALL = pd.DataFrame(sb_rows, columns=['league_season', 'sb_player', 'sb_team']).drop_duplicates()
SB_ALL['sb_key'] = SB_ALL['sb_player'].apply(normalize_name)
print('SB pairs:', len(SB_ALL), '| players:', SB_ALL['sb_player'].nunique(),
      '| seasons:', SB_ALL['league_season'].nunique())
print(SB_ALL.groupby('league_season')['sb_player'].nunique().to_string())


SB pairs: 11568 | players: 6206 | seasons: 28
league_season
1. Bundesliga 2015/2016       421
1. Bundesliga 2023/2024       460
Champions League 2012/2013     36
Champions League 2013/2014     36
Champions League 2014/2015     36
Champions League 2015/2016     36
Champions League 2016/2017     36
Champions League 2017/2018     36
Champions League 2018/2019     46
Copa America 2024             412
Euro 2020                     618
Euro 2024                     621
La Liga 2012/2013             364
La Liga 2013/2014             367
La Liga 2014/2015             432
La Liga 2015/2016             601
La Liga 2016/2017             444
La Liga 2017/2018             441
La Liga 2018/2019             441
La Liga 2019/2020             476
La Liga 2020/2021             516
Ligue 1 2015/2016             654
Ligue 1 2021/2022             420
Ligue 1 2022/2023             485
Premier League 2015/2016      649
Serie A 2015/2016             700
World Cup 2018                739
World Cup 2022        

## Cell 5 — Bulk Transfermarkt side (one big read)

A single pass over appearances.csv with each row assigned to a season by competition plus date range, then club names resolved. Club links stay season-accurate throughout — the current-club snapshot is never used.

In [5]:
app = pd.read_csv(os.path.join(TM_DIR, 'appearances.csv'),
                  usecols=['player_id', 'player_name', 'player_club_id', 'date', 'competition_id'],
                  parse_dates=['date'])
clubs = pd.read_csv(os.path.join(TM_DIR, 'clubs.csv'), usecols=['club_id', 'name'])
tm_rows = []
for _, r in SCOPE.iterrows():
    sub = app[(app['competition_id'] == r['tm_comp']) & (app['date'] >= r['start']) & (app['date'] <= r['end'])]
    sub = sub[['player_id', 'player_name', 'player_club_id']].drop_duplicates()
    sub = sub.merge(clubs, left_on='player_club_id', right_on='club_id')
    sub['league_season'] = r['label']
    tm_rows.append(sub.rename(columns={'name': 'tm_club'})[['league_season', 'player_id', 'player_name', 'tm_club']])
TM_ALL = pd.concat(tm_rows, ignore_index=True)
TM_ALL['tm_key'] = TM_ALL['player_name'].apply(normalize_name)
print('TM pairs:', len(TM_ALL), '| players:', TM_ALL['player_name'].nunique(),
      '| seasons:', TM_ALL['league_season'].nunique())
print(TM_ALL.groupby('league_season')['player_name'].nunique().to_string())


TM pairs: 13207 | players: 5590 | seasons: 23
league_season
1. Bundesliga 2015/2016       477
1. Bundesliga 2023/2024       494
Champions League 2012/2013    643
Champions League 2013/2014    610
Champions League 2014/2015    521
Champions League 2015/2016    573
Champions League 2016/2017    619
Champions League 2017/2018    619
Champions League 2018/2019    639
La Liga 2012/2013             538
La Liga 2013/2014             537
La Liga 2014/2015             466
La Liga 2015/2016             536
La Liga 2016/2017             539
La Liga 2017/2018             556
La Liga 2018/2019             520
La Liga 2019/2020             535
La Liga 2020/2021             683
Ligue 1 2015/2016             575
Ligue 1 2021/2022             521
Ligue 1 2022/2023             582
Premier League 2015/2016      549
Serie A 2015/2016             551


## Cell 6 — Matching, one season at a time (slowest cell)

One unchanged resolve_sample call per league-season — the same method validated on Spain and England. Outputs: combined matches with method labels, a per-season hit-rate table, and the manual-review table (expected to be small; anything in it is flagged, never silently dropped).

In [6]:
finals, reviews = [], []
for label in SCOPE['label']:
    sb = SB_ALL[SB_ALL['league_season'] == label].drop(columns='league_season')
    tm = TM_ALL[TM_ALL['league_season'] == label].drop(columns='league_season')
    if len(sb) == 0 or len(tm) == 0:
        print('SKIP (empty side):', label)
        continue
    final, still = resolve_sample(sb, tm)
    final['league_season'] = label
    finals.append(final)
    if len(still):
        reviews.append((label, still.tolist()))
    print(f'{label}: {len(final)}/{sb["sb_player"].nunique()} '
          f'{final["method"].value_counts().to_dict()} still={len(still)}')
FINAL = pd.concat(finals, ignore_index=True)
print()
print('TOTAL:', len(FINAL))
print('manual-review queue:', reviews if reviews else 'EMPTY')


1. Bundesliga 2023/2024: 401/460 {'exact': 334, 'token': 63, 'fuzzy': 4} still=0
1. Bundesliga 2015/2016: 396/421 {'exact': 322, 'token': 64, 'fuzzy': 10} still=0
Champions League 2018/2019: 41/46 {'exact': 25, 'token': 15, 'nickname': 1} still=0
Champions League 2017/2018: 34/36 {'exact': 16, 'token': 15, 'nickname': 2, 'fuzzy': 1} still=0
Champions League 2016/2017: 34/36 {'exact': 16, 'token': 15, 'nickname': 2, 'fuzzy': 1} still=0
Champions League 2015/2016: 31/36 {'token': 21, 'exact': 6, 'nickname': 3, 'fuzzy': 1} still=0
Champions League 2014/2015: 30/36 {'token': 16, 'exact': 14} still=0
Champions League 2013/2014: 33/36 {'token': 25, 'exact': 6, 'nickname': 2} still=2
Champions League 2012/2013: 33/36 {'exact': 24, 'token': 7, 'fuzzy': 2} still=0
La Liga 2020/2021: 382/516 {'token': 289, 'exact': 83, 'fuzzy': 6, 'nickname': 4} still=1
La Liga 2019/2020: 371/476 {'token': 287, 'exact': 72, 'fuzzy': 8, 'nickname': 4} still=1
La Liga 2018/2019: 360/441 {'token': 275, 'exact': 74,

## Cell 7 — Check: new map and nickname proposals

Rebuilds candidate pairs per season to list club mismatches (same-club spelling gaps propose map additions; cross-club rows stay rejected) and ranks unmatched players by squad appearances (frequent misses propose nickname additions, as the Premier League follow-up showed). Proposals only — the module changes separately, with tests.

In [7]:
from src.join.match import candidate_pairs
for label in SCOPE['label']:
    sb = SB_ALL[SB_ALL['league_season'] == label].drop(columns='league_season')
    tm = TM_ALL[TM_ALL['league_season'] == label].drop(columns='league_season')
    if len(sb) == 0 or len(tm) == 0:
        continue
    pairs = candidate_pairs(sb, tm)
    mm = pairs.loc[~pairs['club_ok'], ['sb_team', 'tm_club']].drop_duplicates()
    same_clubish = mm[mm.apply(lambda r: normalize_name(r['sb_team'])[:6] == normalize_name(r['tm_club'])[:6], axis=1)]
    if len(same_clubish):
        print('## map candidates:', label)
        print(same_clubish.to_string())
unmatched = SB_ALL.loc[~SB_ALL['sb_player'].isin(FINAL['sb_player'])]
freq = unmatched.groupby(['league_season', 'sb_player', 'sb_team']).size().reset_index(name='squads')
print()
print('top unmatched by squad inclusions (nickname-mining list):')
print(freq.sort_values('squads', ascending=False).head(20).to_string())


## map candidates: 1. Bundesliga 2015/2016
                         sb_team                   tm_club
23404   Borussia Mönchengladbach         Borussia Dortmund
198684         Borussia Dortmund  Borussia Mönchengladbach
## map candidates: Ligue 1 2022/2023
                       sb_team              tm_club
75225           Stade Brestois          Stade Reims
77822   Olympique de Marseille  Olympique Marseille
101822          Stade de Reims          Stade Reims
## map candidates: Ligue 1 2021/2022
                       sb_team              tm_club
36551           Stade de Reims          Stade Reims
108388  Olympique de Marseille  Olympique Marseille
## map candidates: Ligue 1 2015/2016
                      sb_team              tm_club
10965  Olympique de Marseille  Olympique Marseille
26743     Stade Malherbe Caen          Stade Reims
78563          Stade de Reims          Stade Reims

top unmatched by squad inclusions (nickname-mining list):
              league_season               

## Cell 8 — Prices and the player-season-price table

Latest price on or before each season's end attaches as the target variable; January movers keep one row per player-team with split_season. The table saves to data/processed/ (gitignored; plain CSV, no new software needed). This file is the handover's core deliverable: every later step (stats, database, modeling) reads it.

In [8]:
movers = FINAL.groupby(['league_season', 'sb_player'])['sb_team'].nunique() > 1
FINAL['split_season'] = FINAL.set_index(['league_season', 'sb_player']).index.map(movers).values
val = pd.read_csv(os.path.join(TM_DIR, 'player_valuations.csv'), parse_dates=['date'])
ends = SCOPE.set_index('label')['end'].to_dict()
parts = []
for label, end in ends.items():
    v = val[val['date'] <= end].sort_values('date').drop_duplicates('player_id', keep='last')
    f = FINAL[FINAL['league_season'] == label].merge(
        v[['player_id', 'date', 'market_value_in_eur']], on='player_id', how='left')
    parts.append(f)
OUT = pd.concat(parts, ignore_index=True)
os.makedirs(OUT_DIR, exist_ok=True)
out_fp = os.path.join(OUT_DIR, 'player_season_value.csv')
OUT.to_csv(out_fp, index=False)
print('rows:', len(OUT), '| with valuation:', OUT['market_value_in_eur'].notna().sum())
print('methods:', OUT['method'].value_counts().to_dict())
print('wrote:', out_fp)
print(OUT.head(10).to_string())


rows: 6454 | with valuation: 6400
methods: {'token': 3291, 'exact': 3009, 'fuzzy': 88, 'nickname': 66}
wrote: c:\Users\User\Development\football-recruitment-intelligence\data\processed\player_season_value.csv
                   sb_player                   sb_team player_id          player_name                   tm_club method            league_season  split_season       date  market_value_in_eur
0                Adam Hložek          Bayer Leverkusen    552057          Adam Hlozek       Bayer 04 Leverkusen  exact  1. Bundesliga 2023/2024         False 2024-05-29           12000000.0
1              Alassane Pléa  Borussia Mönchengladbach    167329        Alassane Pléa  Borussia Mönchengladbach  exact  1. Bundesliga 2023/2024         False 2024-05-29            7000000.0
2  Alejandro Grimaldo García          Bayer Leverkusen    193082   Alejandro Grimaldo       Bayer 04 Leverkusen  token  1. Bundesliga 2023/2024         False 2024-05-29           45000000.0
3        Aleksandar Pavlović   

## Next steps

1. Review Cell 6's manual queue and Cell 7's proposals; apply approved map/nickname additions to src/join/ with tests, then re-run affected seasons.
2. Build per-90 performance stats from events on these joined rows.
3. Design Postgres from this table and load it.
4. Explore the data, then compare models from baseline to XGBoost.